# Phase 3C — Temporal Sequence Construction & Trajectory Analysis

> **Purpose**: Bridge between per-scan embeddings (Phase 3A/B) and downstream consumers (Phase 4 LLM + Phase 5 Video).
>
> **No GPU required** — pure numpy/sklearn/scipy.

### What this notebook does:
1. Load per-scan embeddings (CNN and/or ViT)
2. Group scans by patient → sort by timepoint → build ordered sequences
3. Compute inter-visit metrics (embedding drift, volume changes, growth direction)
4. Generate UMAP trajectory visualizations
5. Save `temporal_sequences.npz` for Phase 4 (LLM narratives) and Phase 5 (video conditioning)

### Inputs (attach as Kaggle datasets):
- `cnn_nnunet_embeddings.npz` and/or `vit_swinunetr_embeddings.npz`
- `tumor_volumes.csv`

### Outputs:
- `temporal_sequences.npz` — per-patient ordered embedding sequences
- `temporal_analysis.json` — summary statistics
- UMAP trajectory plots (saved as PNG)


In [ ]:
import numpy as np
import json as _json
import warnings
import time
import os
from pathlib import Path
from collections import defaultdict

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from scipy.spatial.distance import cosine as cosine_dist
from scipy.stats import spearmanr, kendalltau

warnings.filterwarnings('ignore')
np.random.seed(42)

# ── Output directory ──
OUTPUT_ROOT = Path('/kaggle/working/phase3c_temporal')
FIG_DIR = OUTPUT_ROOT / 'figures'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(exist_ok=True)

# ═══════════════════════════════════════════════════
# LOAD EMBEDDINGS
# ═══════════════════════════════════════════════════
SEARCH_ROOTS = [Path('/kaggle/input'), Path('/kaggle/working')]

models = {}
# Search priority: v3_base > v3 > plain name
# Norm fingerprint: base model mean ~1924, triplet model mean ~1479
for emb_name, model_key in [
    ('vit_swinunetr_embeddings_v3_base', 'swinunetr_base'),
    ('vit_swinunetr_embeddings_v3',      'swinunetr_triplet'),
    ('vit_swinunetr_embeddings',         'swinunetr'),
    ('cnn_nnunet_embeddings',            'nnunet'),
]:
    for root in SEARCH_ROOTS:
        matches = list(root.rglob(f'{emb_name}.npz'))
        if matches:
            data = np.load(matches[0], allow_pickle=True)
            embs = data['embeddings']        # (N, D)
            pids = data['patient_ids']       # (N,)
            tps  = data['timepoints']        # (N,)
            models[model_key] = {
                'embeddings': embs,
                'patient_ids': np.array([str(p) for p in pids]),
                'timepoints': np.array([int(t) for t in tps]),
            }
            norm_mean = float(np.linalg.norm(embs, axis=1).mean())
            id_str = 'BASE (no triplet)' if norm_mean > 1800 else 'TRIPLET-trained'
            print(f'{model_key}: {embs.shape[0]} scans | dim={embs.shape[1]} | norm_mean={norm_mean:.0f} → {id_str}')
            print(f'  File: {matches[0]}')
            break

if not models:
    raise FileNotFoundError('No embeddings found. Attach datasets as input.')

# ── Load tumor volumes ──
tumor_df = None
for root in SEARCH_ROOTS:
    for f in root.rglob('tumor_volumes.csv'):
        import pandas as pd
        tumor_df = pd.read_csv(f)
        tumor_df['patient_id'] = tumor_df['patient_id'].astype(str)
        tumor_df['timepoint'] = tumor_df['timepoint'].astype(int)
        print(f'Tumor volumes: {len(tumor_df)} rows from {f}')
        break
    if tumor_df is not None:
        break

# ── Component-wise L2 normalisation (same as eval notebook) ──
for mn, mdata in models.items():
    arr = mdata['embeddings']
    D = arr.shape[1]
    C_emb = (D - 9) // 11
    n_oct = 8 * C_emb
    n_reg = 3 * C_emb

    comp_oct = arr[:, :n_oct]
    comp_reg = arr[:, n_oct:n_oct+n_reg]
    comp_vol = arr[:, -9:]

    comp_oct_n = comp_oct / (np.linalg.norm(comp_oct, axis=1, keepdims=True) + 1e-8)
    comp_reg_n = comp_reg / (np.linalg.norm(comp_reg, axis=1, keepdims=True) + 1e-8)
    comp_vol_n = comp_vol / (np.linalg.norm(comp_vol, axis=1, keepdims=True) + 1e-8)

    arr_balanced = np.concatenate([comp_oct_n, comp_reg_n, comp_vol_n * 2], axis=1)
    mdata['embeddings_normed'] = arr_balanced
    print(f'{mn}: L2-normalised | C={C_emb} | octant={n_oct} region={n_reg} vol=9 → {arr_balanced.shape[1]}-D')

# Pick best model for main analysis (prefer swinunetr)
MAIN_MODEL = 'swinunetr' if 'swinunetr' in models else list(models.keys())[0]
print(f'\nMain model for temporal analysis: {MAIN_MODEL}')


In [ ]:
# ═══════════════════════════════════════════════════
# BUILD PER-PATIENT TEMPORAL SEQUENCES
# ═══════════════════════════════════════════════════
print('='*60)
print('  BUILDING TEMPORAL SEQUENCES')
print('='*60)

temporal_data = {}  # results for each model

for mn, mdata in models.items():
    embs = mdata['embeddings_normed']
    pids = mdata['patient_ids']
    tps  = mdata['timepoints']
    D    = embs.shape[1]

    # ── Group by patient ──
    patient_visits = defaultdict(list)
    for i in range(len(pids)):
        patient_visits[pids[i]].append({
            'timepoint': tps[i],
            'index': i,
            'embedding': embs[i],
            'raw_embedding': mdata['embeddings'][i],
        })

    # ── Sort each patient's visits by timepoint ──
    for pid in patient_visits:
        patient_visits[pid].sort(key=lambda x: x['timepoint'])

    n_patients = len(patient_visits)
    visit_counts = [len(v) for v in patient_visits.values()]
    multi_visit = {pid: visits for pid, visits in patient_visits.items() if len(visits) >= 2}

    print(f'\n{mn}:')
    print(f'  Total patients: {n_patients}')
    print(f'  Visits per patient: min={min(visit_counts)} max={max(visit_counts)} '
          f'mean={np.mean(visit_counts):.1f} median={np.median(visit_counts):.0f}')
    print(f'  Multi-visit patients (≥2): {len(multi_visit)} ({len(multi_visit)/n_patients*100:.1f}%)')

    # Visit count distribution
    from collections import Counter
    vc = Counter(visit_counts)
    for n_vis in sorted(vc.keys()):
        print(f'    {n_vis} visits: {vc[n_vis]} patients')

    # ── Compute inter-visit temporal features ──
    all_drifts = []        # embedding L2 distances between consecutive visits
    all_cosine_drifts = [] # 1 - cosine similarity
    all_vol_changes = []   # WT volume changes
    all_et_changes = []    # ET volume changes
    octant_drifts = []     # per-octant L2 drift
    patient_summaries = [] # per-patient summary

    C_emb = (mdata['embeddings'].shape[1] - 9) // 11
    n_oct = 8 * C_emb

    for pid, visits in multi_visit.items():
        patient_drifts = []
        patient_vol = []

        for j in range(1, len(visits)):
            prev_emb = visits[j-1]['embedding']
            curr_emb = visits[j]['embedding']
            prev_raw = visits[j-1]['raw_embedding']
            curr_raw = visits[j]['raw_embedding']

            # Embedding drift (L2 in normalised space)
            drift = float(np.linalg.norm(curr_emb - prev_emb))
            all_drifts.append(drift)
            patient_drifts.append(drift)

            # Cosine distance
            cos_d = float(cosine_dist(curr_emb, prev_emb))
            all_cosine_drifts.append(cos_d)

            # Volume changes (from raw embedding last 9 dims)
            wt_prev = float(np.expm1(prev_raw[-9]))   # log1p → original
            wt_curr = float(np.expm1(curr_raw[-9]))
            et_prev = float(np.expm1(prev_raw[-7]))
            et_curr = float(np.expm1(curr_raw[-7]))
            delta_wt = wt_curr - wt_prev
            delta_et = et_curr - et_prev
            all_vol_changes.append(delta_wt)
            all_et_changes.append(delta_et)
            patient_vol.append({'delta_wt': delta_wt, 'delta_et': delta_et})

            # Per-octant drift (which spatial region changed most?)
            oct_prev = prev_emb[:n_oct].reshape(8, -1)
            oct_curr = curr_emb[:n_oct].reshape(8, -1)
            oct_drift = np.linalg.norm(oct_curr - oct_prev, axis=1)
            octant_drifts.append(oct_drift)

        # Patient summary
        total_drift = float(np.linalg.norm(visits[-1]['embedding'] - visits[0]['embedding']))
        patient_summaries.append({
            'patient_id': pid,
            'n_visits': len(visits),
            'timepoints': [v['timepoint'] for v in visits],
            'total_drift': total_drift,
            'mean_step_drift': float(np.mean(patient_drifts)),
            'max_step_drift': float(np.max(patient_drifts)),
            'is_accelerating': len(patient_drifts) >= 2 and patient_drifts[-1] > patient_drifts[0],
        })

    all_drifts = np.array(all_drifts)
    all_cosine_drifts = np.array(all_cosine_drifts)
    all_vol_changes = np.array(all_vol_changes)
    all_et_changes = np.array(all_et_changes)
    octant_drifts = np.array(octant_drifts)

    print(f'\n  Inter-visit metrics ({len(all_drifts)} transitions):')
    print(f'    L2 drift:     mean={all_drifts.mean():.4f}  std={all_drifts.std():.4f}  '
          f'min={all_drifts.min():.4f}  max={all_drifts.max():.4f}')
    print(f'    Cosine dist:  mean={all_cosine_drifts.mean():.4f}  std={all_cosine_drifts.std():.4f}')
    print(f'    ΔWT volume:   mean={all_vol_changes.mean():.0f}  std={all_vol_changes.std():.0f}')
    print(f'    ΔET volume:   mean={all_et_changes.mean():.0f}  std={all_et_changes.std():.0f}')

    # Drift vs volume change correlation
    if len(all_drifts) > 10:
        abs_vol = np.abs(all_vol_changes)
        rho, pval = spearmanr(all_drifts, abs_vol)
        print(f'    Spearman(drift, |ΔWT|): ρ={rho:.3f}  p={pval:.2e}')
        tau, tp = kendalltau(all_drifts, abs_vol)
        print(f'    Kendall(drift, |ΔWT|):  τ={tau:.3f}  p={tp:.2e}')

    # Most changed octant distribution
    if len(octant_drifts) > 0:
        OCTANT_NAMES = ['inf-ant-L', 'inf-ant-R', 'inf-post-L', 'inf-post-R',
                        'sup-ant-L', 'sup-ant-R', 'sup-post-L', 'sup-post-R']
        most_changed = np.argmax(octant_drifts, axis=1)
        oct_counts = Counter(most_changed)
        print(f'\n  Most-changed octant distribution:')
        for oi in range(8):
            pct = oct_counts.get(oi, 0) / len(most_changed) * 100
            bar = '█' * int(pct / 2)
            print(f'    {OCTANT_NAMES[oi]:>12s}: {oct_counts.get(oi,0):4d} ({pct:5.1f}%) {bar}')

    # Accelerating vs decelerating patients
    n_acc = sum(1 for s in patient_summaries if s['is_accelerating'])
    print(f'\n  Trajectory direction ({len(patient_summaries)} multi-visit patients):')
    print(f'    Accelerating (drift increasing): {n_acc} ({n_acc/max(len(patient_summaries),1)*100:.1f}%)')
    print(f'    Decelerating (drift decreasing): {len(patient_summaries)-n_acc}')

    temporal_data[mn] = {
        'patient_visits': patient_visits,
        'multi_visit': multi_visit,
        'all_drifts': all_drifts,
        'all_cosine_drifts': all_cosine_drifts,
        'all_vol_changes': all_vol_changes,
        'all_et_changes': all_et_changes,
        'octant_drifts': octant_drifts,
        'patient_summaries': patient_summaries,
    }


In [ ]:
# ═══════════════════════════════════════════════════
# UMAP TRAJECTORY VISUALIZATIONS
# ═══════════════════════════════════════════════════
print('='*60)
print('  UMAP TRAJECTORY VISUALIZATIONS')
print('='*60)

try:
    from umap import UMAP
    HAS_UMAP = True
except ImportError:
    try:
        import subprocess, sys
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'umap-learn', '-q'])
        from umap import UMAP
        HAS_UMAP = True
    except Exception:
        HAS_UMAP = False
        print('UMAP not available — using t-SNE fallback')

if not HAS_UMAP:
    from sklearn.manifold import TSNE

for mn in models:
    mdata = models[mn]
    tdata = temporal_data[mn]
    embs = mdata['embeddings_normed']
    pids = mdata['patient_ids']
    tps  = mdata['timepoints']

    print(f'\n  {mn}: Computing 2D projection for {len(embs)} scans...')
    t0 = time.time()

    if HAS_UMAP:
        reducer = UMAP(n_components=2, n_neighbors=30, min_dist=0.3, random_state=42)
    else:
        reducer = TSNE(n_components=2, perplexity=30, random_state=42)
    coords_2d = reducer.fit_transform(embs)
    print(f'    Done in {time.time()-t0:.1f}s')

    # ── Plot 1: All scans coloured by timepoint ──
    fig, axes = plt.subplots(1, 2, figsize=(20, 8))

    ax = axes[0]
    tp_norm = (tps - tps.min()) / max(tps.max() - tps.min(), 1)
    scatter = ax.scatter(coords_2d[:, 0], coords_2d[:, 1],
                         c=tp_norm, cmap='viridis', s=8, alpha=0.6)
    plt.colorbar(scatter, ax=ax, label='Timepoint (normalised)')
    ax.set_title(f'{mn} — All scans coloured by timepoint', fontsize=13)
    ax.set_xlabel('UMAP-1'); ax.set_ylabel('UMAP-2')

    # ── Plot 2: Trajectory arrows for multi-visit patients ──
    ax = axes[1]
    ax.scatter(coords_2d[:, 0], coords_2d[:, 1], c='lightgray', s=4, alpha=0.3)

    multi = tdata['multi_visit']
    # Pick up to 50 patients with largest total drift for visibility
    summaries = sorted(tdata['patient_summaries'], key=lambda x: x['total_drift'], reverse=True)
    top_patients = [s['patient_id'] for s in summaries[:50]]

    colors = cm.tab20(np.linspace(0, 1, min(20, len(top_patients))))
    for ci, pid in enumerate(top_patients):
        visits = multi[pid]
        indices = [v['index'] for v in visits]
        pts = coords_2d[indices]
        color = colors[ci % len(colors)]
        ax.plot(pts[:, 0], pts[:, 1], '-', color=color, alpha=0.7, linewidth=1.5)
        ax.scatter(pts[0, 0], pts[0, 1], c=[color], s=40, marker='o', edgecolors='black', linewidths=0.5, zorder=5)
        ax.scatter(pts[-1, 0], pts[-1, 1], c=[color], s=60, marker='*', edgecolors='black', linewidths=0.5, zorder=5)
        # Arrow from second-to-last to last
        if len(pts) >= 2:
            dx = pts[-1, 0] - pts[-2, 0]
            dy = pts[-1, 1] - pts[-2, 1]
            ax.annotate('', xy=(pts[-1, 0], pts[-1, 1]),
                        xytext=(pts[-2, 0], pts[-2, 1]),
                        arrowprops=dict(arrowstyle='->', color=color, lw=1.5))

    ax.set_title(f'{mn} — Top 50 trajectories (● start → ★ end)', fontsize=13)
    ax.set_xlabel('UMAP-1'); ax.set_ylabel('UMAP-2')

    plt.tight_layout()
    out = FIG_DIR / f'{mn}_umap_trajectories.png'
    plt.savefig(out, dpi=150, bbox_inches='tight'); plt.close()
    print(f'    Saved: {out.name}')

    # ── Plot 3: Drift distribution ──
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    ax = axes[0]
    ax.hist(tdata['all_drifts'], bins=50, color='steelblue', edgecolor='white', alpha=0.8)
    ax.axvline(tdata['all_drifts'].mean(), color='red', ls='--', label=f'mean={tdata["all_drifts"].mean():.3f}')
    ax.set_xlabel('L2 Drift'); ax.set_ylabel('Count')
    ax.set_title('Inter-visit embedding drift'); ax.legend()

    ax = axes[1]
    ax.scatter(np.abs(tdata['all_vol_changes']), tdata['all_drifts'],
               s=10, alpha=0.4, c='darkorange')
    ax.set_xlabel('|ΔWT volume|'); ax.set_ylabel('Embedding drift')
    ax.set_title('Drift vs Volume Change')

    ax = axes[2]
    ax.scatter(np.abs(tdata['all_et_changes']), tdata['all_drifts'],
               s=10, alpha=0.4, c='crimson')
    ax.set_xlabel('|ΔET volume|'); ax.set_ylabel('Embedding drift')
    ax.set_title('Drift vs ET Change')

    plt.tight_layout()
    out = FIG_DIR / f'{mn}_drift_analysis.png'
    plt.savefig(out, dpi=150, bbox_inches='tight'); plt.close()
    print(f'    Saved: {out.name}')

    # Store coords for saving
    mdata['umap_coords'] = coords_2d


In [ ]:
# ═══════════════════════════════════════════════════
# SAVE TEMPORAL SEQUENCES FOR PHASE 4 + 5
# ═══════════════════════════════════════════════════
print('='*60)
print('  SAVING TEMPORAL SEQUENCES')
print('='*60)

for mn in models:
    mdata = models[mn]
    tdata = temporal_data[mn]
    patient_visits = tdata['patient_visits']
    D = mdata['embeddings_normed'].shape[1]
    D_raw = mdata['embeddings'].shape[1]

    # Find max visits per patient
    all_pids = sorted(patient_visits.keys())
    T_max = max(len(patient_visits[pid]) for pid in all_pids)
    N = len(all_pids)

    print(f'\n  {mn}: {N} patients | T_max={T_max} | D_normed={D} | D_raw={D_raw}')

    # Build padded arrays
    seq_normed = np.zeros((N, T_max, D), dtype=np.float32)
    seq_raw    = np.zeros((N, T_max, D_raw), dtype=np.float32)
    seq_vol    = np.zeros((N, T_max, 9), dtype=np.float32)
    n_visits   = np.zeros(N, dtype=np.int32)
    timepoints = np.zeros((N, T_max), dtype=np.int32)
    patient_ids = np.array(all_pids, dtype=object)

    # Inter-visit deltas (T_max - 1 per patient)
    deltas_l2      = np.zeros((N, max(T_max - 1, 1)), dtype=np.float32)
    deltas_cosine  = np.zeros((N, max(T_max - 1, 1)), dtype=np.float32)
    deltas_wt_vol  = np.zeros((N, max(T_max - 1, 1)), dtype=np.float32)
    deltas_et_vol  = np.zeros((N, max(T_max - 1, 1)), dtype=np.float32)
    most_changed_octant = np.full((N, max(T_max - 1, 1)), -1, dtype=np.int32)

    C_emb = (D_raw - 9) // 11
    n_oct = 8 * C_emb

    for pi, pid in enumerate(all_pids):
        visits = patient_visits[pid]
        n_vis = len(visits)
        n_visits[pi] = n_vis

        for vi, v in enumerate(visits):
            seq_normed[pi, vi] = v['embedding']
            seq_raw[pi, vi]    = v['raw_embedding']
            seq_vol[pi, vi]    = v['raw_embedding'][-9:]
            timepoints[pi, vi] = v['timepoint']

        # Compute deltas
        for vi in range(1, n_vis):
            prev = visits[vi-1]['embedding']
            curr = visits[vi]['embedding']
            prev_raw = visits[vi-1]['raw_embedding']
            curr_raw = visits[vi]['raw_embedding']

            deltas_l2[pi, vi-1] = float(np.linalg.norm(curr - prev))
            deltas_cosine[pi, vi-1] = float(cosine_dist(curr, prev))
            deltas_wt_vol[pi, vi-1] = float(np.expm1(curr_raw[-9]) - np.expm1(prev_raw[-9]))
            deltas_et_vol[pi, vi-1] = float(np.expm1(curr_raw[-7]) - np.expm1(prev_raw[-7]))

            # Most changed octant
            oct_prev = prev[:n_oct].reshape(8, -1)
            oct_curr = curr[:n_oct].reshape(8, -1)
            oct_drift = np.linalg.norm(oct_curr - oct_prev, axis=1)
            most_changed_octant[pi, vi-1] = int(np.argmax(oct_drift))

    # ── Save ──
    out_path = OUTPUT_ROOT / f'{mn}_temporal_sequences.npz'
    np.savez_compressed(out_path,
        # Per-patient sequences (padded)
        sequences_normed=seq_normed,     # (N, T_max, D)
        sequences_raw=seq_raw,           # (N, T_max, D_raw)
        vol_sequences=seq_vol,           # (N, T_max, 9)
        n_visits=n_visits,               # (N,)
        timepoints=timepoints,           # (N, T_max)
        patient_ids=patient_ids,         # (N,)
        # Inter-visit deltas
        deltas_l2=deltas_l2,             # (N, T_max-1)
        deltas_cosine=deltas_cosine,     # (N, T_max-1)
        deltas_wt_vol=deltas_wt_vol,     # (N, T_max-1)
        deltas_et_vol=deltas_et_vol,     # (N, T_max-1)
        most_changed_octant=most_changed_octant,  # (N, T_max-1)
    )

    size_mb = os.path.getsize(out_path) / 1e6
    print(f'  Saved: {out_path.name}  ({size_mb:.1f} MB)')
    print(f'    sequences_normed: {seq_normed.shape}')
    print(f'    sequences_raw:    {seq_raw.shape}')
    print(f'    vol_sequences:    {seq_vol.shape}')
    print(f'    n_visits:         {n_visits.shape} (min={n_visits.min()} max={n_visits.max()})')
    print(f'    deltas_l2:        {deltas_l2.shape}')

    # ── Save analysis summary ──
    summary = {
        'model': mn,
        'n_patients': int(N),
        'n_scans': int(mdata['embeddings'].shape[0]),
        'embedding_dim_raw': int(D_raw),
        'embedding_dim_normed': int(D),
        'T_max': int(T_max),
        'C_emb': int(C_emb),
        'multi_visit_patients': int(len(tdata['multi_visit'])),
        'mean_visits': float(np.mean(n_visits)),
        'drift_stats': {
            'mean': float(tdata['all_drifts'].mean()),
            'std': float(tdata['all_drifts'].std()),
            'min': float(tdata['all_drifts'].min()),
            'max': float(tdata['all_drifts'].max()),
        },
        'cosine_dist_stats': {
            'mean': float(tdata['all_cosine_drifts'].mean()),
            'std': float(tdata['all_cosine_drifts'].std()),
        },
    }

    json_path = OUTPUT_ROOT / f'{mn}_temporal_analysis.json'
    with open(json_path, 'w') as f:
        _json.dump(summary, f, indent=2)
    print(f'  Analysis: {json_path.name}')


In [ ]:
# ═══════════════════════════════════════════════════
# CROSS-MODEL TEMPORAL COMPARISON
# ═══════════════════════════════════════════════════
print('='*60)
print('  TEMPORAL ANALYSIS SUMMARY')
print('='*60)

if len(models) >= 2:
    print('\n  ── CNN vs ViT Temporal Sensitivity ──')
    print(f'  {"Metric":<30}', end='')
    for mn in sorted(models.keys()):
        print(f' {mn:>12}', end='')
    print('  Winner')
    print('  ' + '─'*65)

    metrics = [
        ('Mean L2 drift', lambda mn: temporal_data[mn]['all_drifts'].mean()),
        ('Drift std', lambda mn: temporal_data[mn]['all_drifts'].std()),
        ('Mean cosine dist', lambda mn: temporal_data[mn]['all_cosine_drifts'].mean()),
        ('Spearman(drift,|ΔWT|)', lambda mn: abs(spearmanr(temporal_data[mn]['all_drifts'],
                                    np.abs(temporal_data[mn]['all_vol_changes']))[0])),
        ('Kendall(drift,|ΔWT|)', lambda mn: abs(kendalltau(temporal_data[mn]['all_drifts'],
                                   np.abs(temporal_data[mn]['all_vol_changes']))[0])),
        ('% accelerating', lambda mn: sum(1 for s in temporal_data[mn]['patient_summaries']
                                        if s['is_accelerating']) / max(len(temporal_data[mn]['patient_summaries']),1)),
    ]

    for name, fn in metrics:
        vals = {}
        for mn in sorted(models.keys()):
            try:
                vals[mn] = fn(mn)
            except Exception:
                vals[mn] = 0.0
        row = f'  {name:<30}'
        for mn in sorted(models.keys()):
            row += f' {vals[mn]:>12.4f}'
        best = max(vals, key=vals.get)
        row += f'  ← {best}'
        print(row)

# Single model summary
for mn in models:
    tdata = temporal_data[mn]
    print(f'\n  {mn} Summary:')
    print(f'    Multi-visit patients: {len(tdata["multi_visit"])}')
    print(f'    Total transitions: {len(tdata["all_drifts"])}')
    print(f'    Mean drift: {tdata["all_drifts"].mean():.4f}')
    print(f'    Mean cosine dist: {tdata["all_cosine_drifts"].mean():.4f}')

    # Top 5 most changed patients
    top5 = sorted(tdata['patient_summaries'], key=lambda x: x['total_drift'], reverse=True)[:5]
    print(f'\n    Top 5 most-changed patients:')
    for s in top5:
        tps_str = '→'.join(str(t) for t in s['timepoints'])
        print(f'      {s["patient_id"]}: drift={s["total_drift"]:.4f} '
              f'visits={s["n_visits"]} ({tps_str}) '
              f'{"↑accel" if s["is_accelerating"] else "↓decel"}')

    # Bottom 5 most stable patients
    bot5 = sorted(tdata['patient_summaries'], key=lambda x: x['total_drift'])[:5]
    print(f'\n    Top 5 most-stable patients:')
    for s in bot5:
        tps_str = '→'.join(str(t) for t in s['timepoints'])
        print(f'      {s["patient_id"]}: drift={s["total_drift"]:.4f} '
              f'visits={s["n_visits"]} ({tps_str})')

print(f'\n  Output directory: {OUTPUT_ROOT}')
print(f'  Figures: {list(FIG_DIR.glob("*.png"))}')
print(f'\n  ✅ Temporal sequences ready for Phase 4 (LLM) and Phase 5 (Video)')
print(f'  Next: Phase 4 — feed sequences + metadata to LLM for clinical narratives')
